# VLA Manipulation Pipeline - Google Colab Verification & Execution Notebook

This notebook runs Phase 0 verification, task simulation, demo collection, SmolVLA fine-tuning, and evaluation.

## 1. Confirm GPU + CUDA

In [ ]:
import torch
print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Please go to Runtime -> Change runtime type -> T4/A100 GPU.")

## 2. Install & Verify LeRobot CLI (with optional dataset/av dependencies)

In [ ]:
!pip install -q "lerobot[dataset]" av
!lerobot-train --help

## 3. Install & Verify ManiSkill3 (Headless EGL Rendering)

In [ ]:
# Install ManiSkill3 dependencies
!pip install -q --no-deps mani_skill
!pip install -q gymnasium h5py trimesh transforms3d pandas sapien dacite GitPython tyro

import os
import torch
import mani_skill.envs
import gymnasium as gym
import matplotlib.pyplot as plt

# Ensure headless EGL rendering configuration for Colab
os.environ["SAPIEN_RENDER_ENGINE"] = "EGL"

# Instantiate environment
env = gym.make("PickCube-v1", obs_mode="rgbd", render_mode="rgb_array")
obs, _ = env.reset()

# Step environment with random action
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

# Render frame to verify visual output
frame = env.render()
if isinstance(frame, list):
    frame = frame[0]
if isinstance(frame, torch.Tensor):
    frame = frame.cpu().numpy()
if frame.ndim == 4:
    frame = frame[0]

plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.title("ManiSkill3 PickCube-v1 Verification Frame")
plt.axis("off")
plt.savefig("maniskill_verification_frame.png")
print("Frame successfully rendered and saved to maniskill_verification_frame.png")
env.close()

## 4. Environment Reset Position Sampling Inspection

Inspects actual object starting XY positions and goal positions sampled across seeds.

In [ ]:
import gymnasium as gym
import mani_skill.envs
import torch
import numpy as np

env = gym.make("PickCube-v1", obs_mode="state_dict")

print("=== In-Distribution Suite (Seeds 1000-1004) ===")
for seed in range(1000, 1005):
    obs, info = env.reset(seed=seed)
    obj_pos = obs["extra"]["obj_pose"][0].cpu().numpy()[:3] if isinstance(obs["extra"]["obj_pose"], torch.Tensor) else obs["extra"]["obj_pose"][:3]
    goal_pos = obs["extra"]["goal_pos"][0].cpu().numpy()[:3] if isinstance(obs["extra"]["goal_pos"], torch.Tensor) else obs["extra"]["goal_pos"][:3]
    print(f"Seed {seed}: Obj Start XY = ({obj_pos[0]:+.4f}, {obj_pos[1]:+.4f}) | Goal XY = ({goal_pos[0]:+.4f}, {goal_pos[1]:+.4f})")

print("\n=== Generalization Suite (Seeds 2000-2004) ===")
for seed in range(2000, 2005):
    obs, info = env.reset(seed=seed)
    obj_pos = obs["extra"]["obj_pose"][0].cpu().numpy()[:3] if isinstance(obs["extra"]["obj_pose"], torch.Tensor) else obs["extra"]["obj_pose"][:3]
    goal_pos = obs["extra"]["goal_pos"][0].cpu().numpy()[:3] if isinstance(obs["extra"]["goal_pos"], torch.Tensor) else obs["extra"]["goal_pos"][:3]
    print(f"Seed {seed}: Obj Start XY = ({obj_pos[0]:+.4f}, {obj_pos[1]:+.4f}) | Goal XY = ({goal_pos[0]:+.4f}, {goal_pos[1]:+.4f})")

env.close()

## 5. Phase 1: Scripted Baseline Policy Verification & 20-Trial Benchmark

In [ ]:
import numpy as np
import torch
import gymnasium as gym
import mani_skill.envs

class ScriptedPickPlacePolicy:
    def __init__(self, env):
        self.env = env
        self.reset()
        
    def reset(self):
        self.stage = 0
        self.step_count = 0
        self.grasped_consecutive_steps = 0
        
    def get_action(self, obs, info=None):
        self.step_count += 1
        
        def to_np(val):
            if isinstance(val, torch.Tensor):
                val = val.cpu().numpy()
            if val.ndim > 1:
                val = val[0]
            return val
            
        extra = obs["extra"]
        tcp_pos = to_np(extra["tcp_pose"])[:3]
        obj_pos = to_np(extra["obj_pose"])[:3]
        goal_pos = to_np(extra["goal_pos"])[:3] if "goal_pos" in extra else obj_pos + np.array([0.0, 0.0, 0.2])
        
        delta_pos = np.zeros(3)
        gripper_action = 1.0
        
        is_grasped = False
        if info is not None and isinstance(info, dict):
            ig = info.get("is_grasped", False)
            if isinstance(ig, torch.Tensor):
                is_grasped = ig.any().item()
            else:
                is_grasped = bool(ig)
                
        if is_grasped:
            self.grasped_consecutive_steps += 1
        else:
            self.grasped_consecutive_steps = 0
            
        if self.stage == 0:
            target = obj_pos + np.array([0.0, 0.0, 0.08])
            diff = target - tcp_pos
            if np.linalg.norm(diff) < 0.015 or self.step_count > 30:
                self.stage = 1
                self.step_count = 0
            else:
                delta_pos = diff * 5.0
                
        elif self.stage == 1:
            target = obj_pos + np.array([0.0, 0.0, 0.010])
            diff = target - tcp_pos
            if np.linalg.norm(diff) < 0.010 or self.step_count > 25:
                self.stage = 2
                self.step_count = 0
                self.grasped_consecutive_steps = 0
            else:
                delta_pos = diff * 5.0
                
        elif self.stage == 2:
            target = obj_pos + np.array([0.0, 0.0, 0.010])
            delta_pos = (target - tcp_pos) * 1.5
            gripper_action = -1.0
            
            if self.grasped_consecutive_steps >= 3 or self.step_count > 30:
                self.stage = 3
                self.step_count = 0
                
        elif self.stage == 3:
            target = np.array([tcp_pos[0], tcp_pos[1], 0.25])
            diff = target - tcp_pos
            gripper_action = -1.0
            lift_gain = 1.5 if self.step_count < 10 else 4.0
            delta_pos = diff * lift_gain
            
            if tcp_pos[2] > 0.20 or self.step_count > 40:
                self.stage = 4
                self.step_count = 0
                
        elif self.stage == 4:
            target = goal_pos
            diff = target - tcp_pos
            gripper_action = -1.0
            if np.linalg.norm(diff) < 0.02 or self.step_count > 45:
                self.stage = 5
                self.step_count = 0
            else:
                delta_pos = diff * 4.0
                
        elif self.stage == 5:
            delta_pos = np.zeros(3)
            gripper_action = -1.0 if self.step_count < 10 else 1.0
            
        delta_pos = np.clip(delta_pos, -1.0, 1.0)
        action = np.array([delta_pos[0], delta_pos[1], delta_pos[2], 0.0, 0.0, 0.0, gripper_action], dtype=np.float32)
        
        if isinstance(extra["tcp_pose"], torch.Tensor) and extra["tcp_pose"].ndim > 1:
            action = torch.tensor(action, device=extra["tcp_pose"].device).unsqueeze(0)
            
        return action

def eval_suite(seed_list, suite_name):
    env = gym.make("PickCube-v1", obs_mode="state_dict", control_mode="pd_ee_delta_pose")
    policy = ScriptedPickPlacePolicy(env)
    successes = 0
    grasps = 0
    print(f"\n=== Evaluating {suite_name} Suite ({len(seed_list)} Trials) ===")
    for i, seed in enumerate(seed_list):
        obs, info = env.reset(seed=seed)
        policy.reset()
        episode_success = False
        made_grasp = False
        for step in range(160):
            action = policy.get_action(obs, info)
            obs, reward, terminated, truncated, info = env.step(action)
            if isinstance(info, dict):
                if info.get("is_grasped", False) or (isinstance(info.get("is_grasped"), torch.Tensor) and info["is_grasped"].any()):
                    made_grasp = True
                if info.get("success", False) or (isinstance(info.get("success"), torch.Tensor) and info["success"].any()):
                    episode_success = True
                    break
        if made_grasp: grasps += 1
        if episode_success: successes += 1
        print(f"[{suite_name}] Trial {i+1:02d}/{len(seed_list):02d} (Seed {seed}): Success={episode_success}, Grasped={made_grasp}")
    env.close()
    print(f"[{suite_name}] Success Rate: {successes/len(seed_list)*100:.1f}% ({successes}/{len(seed_list)})")
    print(f"[{suite_name}] Grasp Success Rate: {grasps/len(seed_list)*100:.1f}% ({grasps}/{len(seed_list)})")
    return successes, grasps

# Run In-Distribution Benchmark (Seeds 1000-1019)
eval_suite(list(range(1000, 1020)), "In-Distribution")

# Run Out-of-Distribution Generalization Benchmark (Seeds 2000-2019)
eval_suite(list(range(2000, 2020)), "Generalization")

## 6. Mounting Google Drive for Persistence

In [ ]:
from google.colab import drive
import os

# Mount Google Drive for persistent checkpoints and results storage
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/vla-manipulation"
os.makedirs(project_dir, exist_ok=True)
print(f"Project persistence directory ready at: {project_dir}")